In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA_DIR = Path.cwd().parent / "data"
DATA_FILE = DATA_DIR / "clusters_report.txt"
registros = []

with open(DATA_FILE, encoding="UTF-8") as archivo:  # apertura del archivo
    lineas = archivo.readlines()                    # lectura linea por linea

for linea in lineas:                                # Inicio del loop
    linea = linea.strip()                           # Se eliminan los espacios existentes entre líneas
    registros.append(linea)                         # Se incluye cada línea en la lista
data = pd.DataFrame(registros, columns =["cluster"])
print(data)                                         # Permite observar un patrón claro: los datos que quiero recuperar no se presentan en todas las líneas pero cuando están siempre aparecenal inicio

                                              cluster
0   Cluster  Cantidad de     Porcentaje de   Princ...
1                      palabras clave  palabras clave
2                                                    
3   ----------------------------------------------...
4   1     105             15,9 %          maximum ...
..                                                ...
63  13    17              2,6 %           pem   fu...
64  optimisation, particles-size  distributions,  ...
65  battery    management,    hydrogen     product...
66                             system-identification.
67                                                   

[68 rows x 1 columns]


In [ ]:
# Realizar el mismo ejercicio realizado previamente: ahora se usará expresiones regulares para leer el apartado requerido de las filas de mi interés: recuperar un primer dato numérico y que vaya hasta el %

import re
import pandas as pd

with open(DATA_FILE, encoding="UTF-8") as archivo:
        lineas = archivo.readlines()

patron = re.compile(r"^(\d+)\s+(\d+)\s+(\d+,\d+)\s*%\s*(.*)$")   # ^la coincidencia debe estar al inicio de la linea: así se excluyen las líneas que no son de interés
                                                                 # Se sigue un esquema de digito y espacio para las columnas de cluster y cantidad de palabras clave: ambas conformadas por números enteros
registros = []                                                   # Para porcentaje se sigue la estrcutura digito , digita %, tal como se visualizó anteriormente en los datos
for linea in lineas:
    linea = linea.strip()                                # Elimina los saltos de línea \n: ahora linea será cada linea del txt, tal como se visualiza
    coincidencia = patron.match(linea)                   # Se busca coincidencias en cada línea que cumpla con la condición del regex
    if coincidencia:
        registros.append(list(coincidencia.groups()))          # Ante cada coincidencia, se introducen todos los elementos de la lineas que coinciden en una liita dentro de la lista registros
        print(registros)                                       # Se evidenica que entre palabras de una misma list hay más de un espacio simple, esto debe corregirse
    elif len(linea) > 0 and len(registros) > 0:                            # Se le dará u tratamiento a todas las líneas no vacías y una vez la lista registros tenga más de una línea que cumpla la condición de patron
        registros[-1][3] += " " + linea                  # Como registros es una lista de listas se define: [-1] llama al último elemento de registros (la última línea leída) y sobre esa última línea leída se aplica [3], el último elemento de la lista que contiene el contenido de la frase que corresponde a las palabras clave
                                                         # El += indica que este indíce 3 de cada lista que cunpla la condición será modificado, añadiendo un espacio y después el contenido de la línea procesada

resultado = pd.DataFrame(registros, columns=["cluster", "cantidad_de_palabras_clave", "porcentaje_de_palabras_clave", "principales_palabras_clave"])   # Se crean las columnas con los nombres requeridos
resultado = resultado.astype({"cluster": int, "cantidad_de_palabras_clave": int})                                             # Se inidca el tipo de datoa a asociar a cada columna 
resultado["porcentaje_de_palabras_clave"] = resultado["porcentaje_de_palabras_clave"].str.replace(",", ".").astype(float)     #Para la columna porcentajee de palabras clave se sustituye la coma por un punto y se convierte a float para reportar el dato en el formato solicitado

# Para ajustar el problema de los espacios se usará una función anónima en lambda en lugar de un ciclo for o de una función def
resultado["principales_palabras_clave"] = resultado["principales_palabras_clave"].apply(                  # Se define como parámetro de la función lambda al parámetro texto
    lambda texto: ", ".join(" ".join(p.split()) for p in texto.split(",") if p.strip()).rstrip("."))      # Primero se hace un split sobre texto, tomando las comas como separador, cada nuevo elemento será llamado p. Luego se eliminan de p todos los espacios existentes (if p.strip)
                                                                                                          # Al aplicar p.split() se crea una lista conformada por acada palabra de p (que representa cada palabra clave) 
                                                                                                          # Luego " ".join() une cada una de estas palabras nuevamente por medio del espacio vacío. Finalmente se retira el último punto final con rstrip()
resultado

[['1', '105', '15,9', 'maximum power point tracking, fuzzy-logic based control, photo voltaic (pv),']]
[['1', '105', '15,9', 'maximum power point tracking, fuzzy-logic based control, photo voltaic (pv), photo-voltaic  system,  differential   evolution   algorithm,   evolutionary algorithm, double-fed induction generator (dfig), ant  colony  optimisation, photo voltaic array, firefly algorithm, partial shade.'], ['2', '102', '15,4', 'support vector machine,  long  short-term  memory,  back-propagation  neural']]
[['1', '105', '15,9', 'maximum power point tracking, fuzzy-logic based control, photo voltaic (pv), photo-voltaic  system,  differential   evolution   algorithm,   evolutionary algorithm, double-fed induction generator (dfig), ant  colony  optimisation, photo voltaic array, firefly algorithm, partial shade.'], ['2', '102', '15,4', 'support vector machine,  long  short-term  memory,  back-propagation  neural network,  convolution  neural  network,  speed   wind   prediction,   en

,cluster,cantidad_de_palabras_clave,porcentaje_de_palabras_clave,principales_palabras_clave
0,1,105,15.9,"maximum power point tracking, fuzzy-logic base..."
1,2,102,15.4,"support vector machine, long short-term memory..."
2,3,89,13.4,"smart grid, wind power, reinforcement learning..."
3,4,60,9.1,"wind turbine, fault diagnosis, biodiesel, fail..."
4,5,52,7.9,"electric vehicle, lithium-ion batteries, state..."
5,6,51,7.7,"particle swarm optimization, distribute genera..."
6,7,42,6.3,"multi-objective optimization, energy storage, ..."
7,8,38,5.7,"genetic algorithm, demand-side management, ene..."
8,9,35,5.3,"anfis, global solar irradiance, solar irradian..."
9,10,27,4.1,"micro grid, multi-agent systems, distributed e..."
